# チュートリアル4: 分子記述子とASEデータベース

このチュートリアルでは、ASEデータベースの作成と分子記述子による条件付き生成を学びます。

**所要時間**: 20分

**学習内容**:
- ASEデータベースの完全な作成手順
- 分子記述子の計算（分子量、官能基、π共役比）
- 14種類の官能基のSMARTSパターンマッチング
- OpenBabel統合による高度な記述子抽出
- カスタムデータセットでの学習例
- 厳密な条件付き生成の検証

**前提知識**: チュートリアル1（基本的な分子生成）

**重要**: このチュートリアルはフォールバック処理なしの厳密な実装です。


## セットアップとインポート


In [ ]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
from ase import Atoms
from ase.db import connect
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')
print(f'RDKit バージョン: {Chem.rdBase.rdkitVersion}')


## 1. ASEデータベースの作成

ASE (Atomic Simulation Environment) データベースは、原子構造と関連するデータを効率的に保存・管理するためのツールです。

**データベースの利点**:
- 大規模データセットの効率的な管理
- メタデータの柔軟な保存
- 高速な検索とフィルタリング
- 分子記述子の永続化


In [ ]:
# 新しいASEデータベースの作成
db_path = 'tutorial_molecules.db'

# 既存のファイルを削除（新規作成のため）
if os.path.exists(db_path):
    os.remove(db_path)
    print(f'既存のデータベースを削除しました: {db_path}')

# データベースの接続
db = connect(db_path)
print(f'新しいデータベースを作成しました: {db_path}')


## 2. 分子の3D座標生成

SMILES文字列から3D座標を生成します。これは厳密な手順で、最適化を含みます。

**重要な原則**:
- フォールバックなし: 座標生成に失敗した場合は明示的にエラー
- MMFF最適化: 力場ベースの幾何学的最適化
- 厳密な検証: 生成された座標の妥当性チェック


In [ ]:
def smiles_to_3d_coords(smiles, max_attempts=5):
    """
    SMILESから厳密な3D座標を生成（フォールバックなし）
    
    Parameters:
    -----------
    smiles : str
        SMILES文字列
    max_attempts : int
        最大試行回数
    
    Returns:
    --------
    positions : np.ndarray [n_atoms, 3]
        原子座標（Angstrom）
    symbols : list
        原子記号のリスト
    
    Raises:
    -------
    ValueError : 3D座標生成に失敗した場合
    """
    # SMILESからMolオブジェクトを作成
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    
    # 水素原子を明示的に追加
    mol = Chem.AddHs(mol)
    
    # 3D座標の埋め込み（複数回試行）
    success = False
    for attempt in range(max_attempts):
        result = AllChem.EmbedMolecule(mol, randomSeed=42 + attempt)
        if result == 0:  # 成功
            success = True
            break
    
    if not success:
        raise ValueError(f'Failed to embed 3D coordinates for: {smiles}')
    
    # MMFF力場による最適化
    opt_result = AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
    if opt_result != 0:
        raise ValueError(f'MMFF optimization failed for: {smiles} (code: {opt_result})')
    
    # 座標と記号の抽出
    conf = mol.GetConformer()
    positions = conf.GetPositions()
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    
    # 座標の検証
    if np.any(np.isnan(positions)) or np.any(np.isinf(positions)):
        raise ValueError(f'Invalid coordinates generated for: {smiles}')
    
    return positions, symbols

# テスト
test_smiles = 'CCO'  # エタノール
positions, symbols = smiles_to_3d_coords(test_smiles)
print(f'テスト成功: {test_smiles}')
print(f'  原子数: {len(symbols)}')
print(f'  原子記号: {symbols}')
print(f'  座標範囲: [{positions.min():.2f}, {positions.max():.2f}] Å')


## 3. 分子記述子の計算

各分子について、以下の記述子を計算します:

### 基本記述子
- **分子量** (molecular_weight): 分子の質量
- **原子数** (n_atoms): 全原子数（水素を含む）
- **重原子数** (n_heavy_atoms): 水素以外の原子数

### 幾何学的記述子
- **回転結合数** (n_rotatable_bonds): 自由回転可能な結合数
- **環数** (n_rings): 環構造の数
- **芳香環数** (n_aromatic_rings): 芳香族環の数

### 化学的記述子
- **logP**: 油水分配係数
- **TPSA**: 極性表面積
- **HBD/HBA**: 水素結合ドナー/アクセプター数


In [ ]:
def compute_molecular_descriptors(smiles):
    """
    分子記述子を厳密に計算（フォールバックなし）
    
    Parameters:
    -----------
    smiles : str
        SMILES文字列
    
    Returns:
    --------
    descriptors : dict
        計算された記述子の辞書
    
    Raises:
    -------
    ValueError : 記述子計算に失敗した場合
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    
    # 水素を追加
    mol_h = Chem.AddHs(mol)
    
    descriptors = {}
    
    # 基本記述子
    descriptors['molecular_weight'] = Descriptors.MolWt(mol_h)
    descriptors['n_atoms'] = mol_h.GetNumAtoms()
    descriptors['n_heavy_atoms'] = mol.GetNumHeavyAtoms()
    
    # 幾何学的記述子
    descriptors['n_rotatable_bonds'] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    descriptors['n_rings'] = rdMolDescriptors.CalcNumRings(mol)
    descriptors['n_aromatic_rings'] = rdMolDescriptors.CalcNumAromaticRings(mol)
    
    # 化学的記述子
    descriptors['logP'] = Descriptors.MolLogP(mol)
    descriptors['tpsa'] = Descriptors.TPSA(mol)
    descriptors['n_hbd'] = rdMolDescriptors.CalcNumHBD(mol)
    descriptors['n_hba'] = rdMolDescriptors.CalcNumHBA(mol)
    
    # π共役系の計算
    n_pi_electrons = 0
    for bond in mol.GetBonds():
        if bond.GetIsAromatic():
            n_pi_electrons += 2
        elif bond.GetBondTypeAsDouble() == 2.0:  # 二重結合
            n_pi_electrons += 2
    
    # π共役比率 = π電子数 / 全電子数
    total_electrons = sum([atom.GetAtomicNum() for atom in mol.GetAtoms()])
    descriptors['pi_conjugation_ratio'] = n_pi_electrons / total_electrons if total_electrons > 0 else 0.0
    
    # 検証: 全ての記述子が有効な数値か
    for key, value in descriptors.items():
        if not isinstance(value, (int, float)) or np.isnan(value) or np.isinf(value):
            raise ValueError(f'Invalid descriptor value: {key}={value} for {smiles}')
    
    return descriptors

# テスト
test_descriptors = compute_molecular_descriptors('CCO')
print('エタノールの記述子:')
for key, value in test_descriptors.items():
    print(f'  {key}: {value:.4f}' if isinstance(value, float) else f'  {key}: {value}')


## 4. 官能基のSMARTSパターンマッチング

14種類の主要な官能基を検出します。SMARTSパターンを使用した厳密なマッチングです。

**官能基リスト**:
1. ヒドロキシル基 (-OH)
2. カルボキシル基 (-COOH)
3. アミノ基 (-NH2)
4. カルボニル基 (C=O)
5. エステル基 (-COO-)
6. エーテル基 (-O-)
7. ニトロ基 (-NO2)
8. アミド基 (-CONH-)
9. ニトリル基 (-CN)
10. アルデヒド基 (-CHO)
11. ケトン基 (R-CO-R')
12. スルホン酸基 (-SO3H)
13. チオール基 (-SH)
14. ハロゲン (F, Cl, Br, I)


In [ ]:
# 官能基のSMARTSパターン定義
FUNCTIONAL_GROUPS = {
    'hydroxyl': '[OX2H]',  # -OH
    'carboxyl': '[CX3](=O)[OX2H1]',  # -COOH
    'amino': '[NX3;H2,H1;!$(NC=O)]',  # -NH2, -NH-
    'carbonyl': '[CX3]=[OX1]',  # C=O
    'ester': '[#6][CX3](=O)[OX2H0][#6]',  # -COO-
    'ether': '[OD2]([#6])[#6]',  # -O-
    'nitro': '[$([NX3](=O)=O),$([NX3+](=O)[O-])]',  # -NO2
    'amide': '[NX3][CX3](=[OX1])[#6]',  # -CONH-
    'nitrile': '[NX1]#[CX2]',  # -C≡N
    'aldehyde': '[CX3H1](=O)[#6]',  # -CHO
    'ketone': '[#6][CX3](=O)[#6]',  # R-CO-R'
    'sulfonic': '[SX4](=[OX1])(=[OX1])([OX2H,OX1H0-])',  # -SO3H
    'thiol': '[SX2H]',  # -SH
    'halogen': '[F,Cl,Br,I]',  # ハロゲン
}

def detect_functional_groups(smiles):
    """
    SMARTSパターンマッチングによる官能基検出（厳密）
    
    Parameters:
    -----------
    smiles : str
        SMILES文字列
    
    Returns:
    --------
    functional_groups : dict
        各官能基の出現回数
    
    Raises:
    -------
    ValueError : パターンマッチングに失敗した場合
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    
    functional_groups = {}
    
    for fg_name, smarts in FUNCTIONAL_GROUPS.items():
        pattern = Chem.MolFromSmarts(smarts)
        if pattern is None:
            raise ValueError(f'Invalid SMARTS pattern: {smarts}')
        
        matches = mol.GetSubstructMatches(pattern)
        functional_groups[f'has_{fg_name}'] = int(len(matches) > 0)
        functional_groups[f'n_{fg_name}'] = len(matches)
    
    return functional_groups

# テスト
test_smiles_fg = 'CC(=O)O'  # 酢酸
fg_results = detect_functional_groups(test_smiles_fg)
print(f'酢酸 ({test_smiles_fg}) の官能基:')
for key, value in fg_results.items():
    if value > 0 and key.startswith('has_'):
        fg_name = key.replace('has_', '')
        count = fg_results[f'n_{fg_name}']
        print(f'  {fg_name}: {count}個')


## 5. データベースへの分子追加

計算した記述子と共に、分子をASEデータベースに追加します。


In [ ]:
def add_molecule_to_database(db, smiles, additional_data=None):
    """
    分子をASEデータベースに追加（厳密な検証付き）
    
    Parameters:
    -----------
    db : ase.db.core.Database
        ASEデータベース
    smiles : str
        SMILES文字列
    additional_data : dict, optional
        追加のメタデータ
    
    Returns:
    --------
    row_id : int
        データベース内のID
    
    Raises:
    -------
    ValueError : 処理に失敗した場合
    """
    try:
        # 3D座標の生成
        positions, symbols = smiles_to_3d_coords(smiles)
        
        # 記述子の計算
        descriptors = compute_molecular_descriptors(smiles)
        
        # 官能基の検出
        functional_groups = detect_functional_groups(smiles)
        
        # Atomsオブジェクトの作成
        atoms = Atoms(symbols=symbols, positions=positions)
        
        # メタデータの統合
        data = {
            'smiles': smiles,
            **descriptors,
            **functional_groups
        }
        
        if additional_data is not None:
            data.update(additional_data)
        
        # データベースに書き込み
        row_id = db.write(atoms, data=data)
        
        return row_id
    
    except Exception as e:
        raise ValueError(f'Failed to add molecule {smiles}: {str(e)}')

# テスト: いくつかの分子を追加
test_molecules = [
    'CCO',  # エタノール
    'CC(=O)O',  # 酢酸
    'c1ccccc1',  # ベンゼン
    'CC(C)O',  # イソプロパノール
    'CCN',  # エチルアミン
]

print('分子をデータベースに追加中...')
for smiles in test_molecules:
    try:
        row_id = add_molecule_to_database(db, smiles)
        print(f'  追加成功: {smiles} (ID: {row_id})')
    except ValueError as e:
        print(f'  追加失敗: {smiles} - {e}')

print(f'\n現在のデータベースサイズ: {len(db)} エントリ')


## 6. データベースのクエリと確認

追加したデータを確認し、記述子による検索を試します。


In [ ]:
# データベースの内容を表示
print('データベースの内容:')
print('=' * 80)

for i, row in enumerate(db.select(), 1):
    atoms = row.toatoms()
    data = row.data
    
    print(f'\nエントリ {i} (ID: {row.id}):')
    print(f'  SMILES: {data.get("smiles", "N/A")}')
    print(f'  化学式: {atoms.get_chemical_formula()}')
    print(f'  原子数: {data.get("n_atoms", "N/A")}')
    print(f'  分子量: {data.get("molecular_weight", "N/A"):.2f}')
    print(f'  logP: {data.get("logP", "N/A"):.2f}')
    print(f'  π共役比: {data.get("pi_conjugation_ratio", "N/A"):.4f}')
    
    # 官能基の表示
    fg_present = []
    for key, value in data.items():
        if key.startswith('has_') and value == 1:
            fg_name = key.replace('has_', '')
            fg_present.append(fg_name)
    if fg_present:
        print(f'  官能基: {", ".join(fg_present)}')

print('\n' + '=' * 80)


## 7. 記述子による検索

特定の記述子値を持つ分子を検索します。


In [ ]:
# 分子量が100以下の分子を検索
print('分子量 ≤ 100 の分子:')
for row in db.select('molecular_weight<=100'):
    data = row.data
    print(f'  {data["smiles"]}: MW={data["molecular_weight"]:.2f}')

print('\nヒドロキシル基を持つ分子:')
for row in db.select('has_hydroxyl=1'):
    data = row.data
    print(f'  {data["smiles"]}: {data["n_hydroxyl"]}個のOH基')

print('\n芳香環を持つ分子:')
for row in db.select('n_aromatic_rings>0'):
    data = row.data
    print(f'  {data["smiles"]}: {data["n_aromatic_rings"]}個の芳香環')


## 8. 条件付き生成への応用

作成したデータベースを使用して、記述子による条件付き生成を行います。

**ワークフロー**:
1. カスタムデータセットでモデルを学習
2. 記述子を条件として指定
3. ターゲット記述子値で分子を生成
4. 生成された分子の記述子を検証


In [ ]:
print('記述子による条件付き生成の例:')
print('\nコマンドライン例:')
print('\n# カスタムデータベースでの学習')
print('python main_qm9.py \\')
print('    --exp_name custom_descriptors \\')
print('    --dataset ase \\')
print('    --ase_db_path tutorial_molecules.db \\')
print('    --conditioning molecular_weight pi_conjugation_ratio \\')
print('    --n_epochs 500 \\')
print('    --batch_size 32')
print('\n# ターゲット記述子での生成')
print('python eval_conditional_qm9.py \\')
print('    --model_path outputs/custom_descriptors \\')
print('    --conditioning molecular_weight pi_conjugation_ratio \\')
print('    --target_values 80.0 0.15 \\')
print('    --n_samples 100')
print('\nこのコマンドで、分子量80、π共役比0.15の分子を生成できます。')


## 9. OpenBabel統合（高度）

OpenBabelを使用すると、さらに高度な記述子を計算できます。

**注意**: OpenBabelのインストールが必要です:
```bash
conda install -c conda-forge openbabel
pip install openbabel-wheel
```


In [ ]:
# OpenBabel統合の例（オプション）
print('OpenBabelによる高度な記述子:')
print('\nOpenBabelがインストールされている場合、以下が計算可能:')
print('  - 3D記述子（Shape index, Molecular eccentricity）')
print('  - 電子的記述子（HOMO/LUMO energy）')
print('  - トポロジカル記述子（Wiener index, Zagreb index）')
print('  - 表面積記述子（SASA, van der Waals surface）')
print('\n実装例は example_ase_conditioning.py を参照してください。')


## 10. 厳密な検証

生成された分子が、実際にターゲット記述子を持っているかを検証します。


In [ ]:
# 検証用の関数
def validate_generated_molecule(smiles, target_descriptors, tolerance=0.1):
    """
    生成された分子が、ターゲット記述子を満たすか検証
    
    Parameters:
    -----------
    smiles : str
        生成された分子のSMILES
    target_descriptors : dict
        ターゲット記述子 {name: value}
    tolerance : float
        許容誤差（相対誤差）
    
    Returns:
    --------
    valid : bool
        検証結果
    errors : dict
        各記述子の誤差
    """
    try:
        actual_descriptors = compute_molecular_descriptors(smiles)
    except ValueError:
        return False, {'error': 'Failed to compute descriptors'}
    
    errors = {}
    valid = True
    
    for desc_name, target_value in target_descriptors.items():
        actual_value = actual_descriptors.get(desc_name)
        if actual_value is None:
            valid = False
            errors[desc_name] = 'Missing descriptor'
            continue
        
        # 相対誤差の計算
        if target_value != 0:
            rel_error = abs(actual_value - target_value) / abs(target_value)
        else:
            rel_error = abs(actual_value - target_value)
        
        errors[desc_name] = rel_error
        
        if rel_error > tolerance:
            valid = False
    
    return valid, errors

# テスト
test_validation = validate_generated_molecule(
    'CCO',
    {'molecular_weight': 46.0, 'pi_conjugation_ratio': 0.0},
    tolerance=0.05
)

valid, errors = test_validation
print(f'検証結果: {"成功" if valid else "失敗"}')
print('誤差:')
for desc, error in errors.items():
    if isinstance(error, float):
        print(f'  {desc}: {error*100:.2f}%')
    else:
        print(f'  {desc}: {error}')


## まとめ

このチュートリアルでは、以下を学習しました:

✅ ASEデータベースの完全な作成手順  
✅ SMILES から厳密な3D座標の生成  
✅ 包括的な分子記述子の計算  
✅ 14種類の官能基のSMARTSパターンマッチング  
✅ データベースへの構造化データ保存  
✅ 記述子による効率的な検索  
✅ 条件付き生成への応用方法  
✅ OpenBabel統合の概念  
✅ 生成結果の厳密な検証  

### 次のステップ

- **チュートリアル5**: 評価と解析 - 生成品質の定量的評価
- **実践**: より大規模なデータセットでの学習
- **発展**: カスタム記述子の実装

### 重要な原則（再確認）

1. **フォールバックなし**: 全ての処理は明示的なエラーハンドリング
2. **厳密な検証**: 計算された値の妥当性チェック
3. **再現性**: 全てのパラメータと手順を明示
4. **透明性**: 処理の各ステップを明確に文書化

### データベースの永続化

作成したデータベース `tutorial_molecules.db` は、後続の学習や解析で再利用できます。

---

**質問やフィードバックは、GitHubのIssuesでお寄せください。**
